### Experiment 3 and 4

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingClassifier, BaggingClassifier, RandomForestRegressor, StackingRegressor, AdaBoostRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

In [10]:
# Load and preprocess Experiment 1 data (MNIST Digits dataset)
digits_data = pd.read_csv("digit.csv")
X_digits = digits_data.iloc[:, 1:].values
y_digits = digits_data.iloc[:, 0].values

In [11]:
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_digits, y_digits, test_size=0.2, random_state=42)

In [12]:
# Load and preprocess Experiment 2 data (Boston Housing dataset)
housing_data = pd.read_csv("house.csv")


In [13]:
housing_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [14]:
# Separate numeric and categorical columns
numeric_columns = housing_data.select_dtypes(include=['float64', 'int64']).columns.drop("SalePrice")
categorical_columns = housing_data.select_dtypes(include=['object']).columns

X_housing = housing_data.drop("SalePrice", axis=1)
y_housing = housing_data["SalePrice"]

In [16]:
# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

X_housing_processed = preprocessor.fit_transform(X_housing)

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_housing_processed, y_housing, test_size=0.2, random_state=42)


In [17]:
# Experiment 3: Ensemble Methods
# Voting Classifier for Experiment 1
voting_clf = VotingClassifier(estimators=[
    ('lr', LogisticRegression(max_iter=1000)),
    ('svm', SVC(probability=True)),
    ('dt', DecisionTreeClassifier())
], voting='soft')

voting_clf.fit(X_train_1, y_train_1)
voting_pred = voting_clf.predict(X_test_1)
print("Voting Classifier Report:")
print(classification_report(y_test_1, voting_pred))

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Voting Classifier Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       816
           1       0.97      0.98      0.98       909
           2       0.95      0.94      0.95       846
           3       0.95      0.93      0.94       937
           4       0.95      0.95      0.95       839
           5       0.94      0.93      0.94       702
           6       0.94      0.97      0.96       785
           7       0.96      0.95      0.96       893
           8       0.94      0.94      0.94       835
           9       0.94      0.94      0.94       838

    accuracy                           0.95      8400
   macro avg       0.95      0.95      0.95      8400
weighted avg       0.95      0.95      0.95      8400



In [19]:
# Bagging Classifier for Experiment 1
bagging_clf = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=10, random_state=42)
bagging_clf.fit(X_train_1, y_train_1)
bagging_pred = bagging_clf.predict(X_test_1)
print("Bagging Classifier Report:")
print(classification_report(y_test_1, bagging_pred))

Bagging Classifier Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       816
           1       0.96      0.98      0.97       909
           2       0.90      0.91      0.91       846
           3       0.93      0.90      0.91       937
           4       0.93      0.93      0.93       839
           5       0.91      0.90      0.90       702
           6       0.96      0.95      0.95       785
           7       0.95      0.94      0.94       893
           8       0.92      0.91      0.91       835
           9       0.91      0.91      0.91       838

    accuracy                           0.93      8400
   macro avg       0.93      0.93      0.93      8400
weighted avg       0.93      0.93      0.93      8400



In [22]:
from sklearn.impute import SimpleImputer

# Update preprocessing pipeline to include imputation for numeric data
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),  # Fill missing values with mean
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

# Preprocess the data
X_housing_processed = preprocessor.fit_transform(X_housing)

# Split the data
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_housing_processed, y_housing, test_size=0.2, random_state=42)


In [23]:
# Random Forest Regressor for Experiment 2
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_2, y_train_2)
rf_pred = rf_regressor.predict(X_test_2)
print("Random Forest Regressor Metrics:")
print(f"MAE: {mean_absolute_error(y_test_2, rf_pred)}")
print(f"MSE: {mean_squared_error(y_test_2, rf_pred)}")
print(f"R2 Score: {r2_score(y_test_2, rf_pred)}")

Random Forest Regressor Metrics:
MAE: 17686.565
MSE: 817435223.9491322
R2 Score: 0.8934288840045431


In [27]:
# Stacking Regressor for Experiment 2
stacking_regressor = StackingRegressor(estimators=[
    ('scaled_svr', Pipeline([('scaler', StandardScaler()), ('svr', SVR())])),  # Wrap SVR with a scaler
    ('dt', DecisionTreeRegressor())
], final_estimator=RandomForestRegressor(random_state=42))

stacking_regressor.fit(X_train_2, y_train_2)
stacking_pred = stacking_regressor.predict(X_test_2)
print("Stacking Regressor Metrics:")
print(f"MAE: {mean_absolute_error(y_test_2, stacking_pred)}")
print(f"MSE: {mean_squared_error(y_test_2, stacking_pred)}")
print(f"R2 Score: {r2_score(y_test_2, stacking_pred)}")



NameError: name 'SVR' is not defined